# Suite PAR — Parametrics

A parameter is a name a definition declares (`Schemas.OfParameter`), with a description: a symbol standing for what is
bound where the definition is referred to. Parameters are `Parametrics`', normal schemas and one extension, installed
before use ([CAP-112: parameters modeled as schema data](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-112.md)): a set of parameters (`Parametrics.OfParameters`), each named by its entry
(`Parametrics.Defines`) and for a value of a schema or an object of an object schema, declared by a definition of any
kind (`parameters`). A schema declares parameters, and a reference object that the schema governs binds them, through
the adjacencies `Parametrics.extend` gives the schema ([CAP-314: parameters bound by the data objects their schema governs](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-314)). Here: installing it, declaring and
validating parameters, snapshots writing them, extending a schema, and binding parameters.

In [ ]:
from mbse.Schemas.Framework import Builders as B, Parametrics, Plain, Schemas as S, Validators
from support import entries_in, given, raises, spec_of

<a id="PAR-01"></a>
## PAR-01 · Parametrics is installed before use: its schemas, a parameter's ends and every kind's set of parameters; installed again, it writes nothing

In [ ]:
KINDS = ("OfScalar", "OfObject", "OfUnion", "OfIntersection", "OfContainer", "OfAny", "OfRelation", "OfPattern",
         "OfExpression", "OfCollector", "OfModule")
installed = B.Store()
assert "Parametrics.OfParameters" not in installed.names()  # not installed: a store has none
assert "for_value" not in installed.Schemas.OfParameter.as_schema().adjacencies
assert "parameters" not in installed.Schemas.OfObject.as_schema().adjacencies  # the core declares none
installed.transaction(Parametrics.install)
assert [name for name in installed.names() if name.startswith("Parametrics.")] == [
    "Parametrics.OfParameters", "Parametrics.Defines", "Parametrics.ForValue", "Parametrics.ForObject",
    "Parametrics.HasParameters", "Parametrics.BindsValue", "Parametrics.BindsObject"]
Defines, ForValue, ForObject, HasParameters, BindsValue, BindsObject = (installed.registered(f"Parametrics.{name}") for name in (
    "Defines", "ForValue", "ForObject", "HasParameters", "BindsValue", "BindsObject"))
assert installed.registered("Parametrics.OfParameters").ref and list(installed.registered("Parametrics.OfParameters").adjacencies) == ["parameters"]
assert list(Defines.entry.links) == ["parameters", "parameter"] and list(Defines.entry.properties) == ["name"]
assert Defines.uniques == (frozenset({"parameters", "name"}), frozenset({"parameter"}))  # keyed by name
assert ForValue.uniques == ForObject.uniques == (frozenset({"schema"}),)  # one schema per parameter, shared by many
assert HasParameters.uniques == (frozenset({"defn"}), frozenset({"params"}))  # unique in both directions
assert list(BindsValue.entry.links) == ["object", "param"] and BindsValue.entry.properties["value"].schema.schemas == ()  # any value
assert list(BindsObject.entry.links) == ["object", "param", "obj_ref"]  # an object binds, not a schema
OfParameter = installed.Schemas.OfParameter.as_schema()  # the extension
assert [(name, a.relation, a.me) for name, a in OfParameter.adjacencies.items()][-2:] == [
    ("for_value", ForValue, "param"), ("for_object", ForObject, "param")]
assert [(a.relation, a.me) for a in OfParameter.anonymous][-3:] == [
    (Defines, "parameter"), (BindsValue, "param"), (BindsObject, "param")]  # what defines it, and what binds it
for kind in KINDS:
    held = getattr(installed.Schemas, kind).as_schema().adjacencies["parameters"]
    assert held.relation is HasParameters and held.me == "defn"
log = Plain.ToPlain(installed).Snapshot()
installed.transaction(Parametrics.install)  # installed again: found, nothing written
assert Plain.ToPlain(installed).Snapshot() == log
assert "Parametrics.OfParameters" not in B.Store().names()  # each store's own

<a id="PAR-02"></a>
## PAR-02 · Every kind of schema declares a set of parameters, each named by its entry, described, and for a value of a schema or an object of an object schema

In [ ]:
def declaring(txn):
    """A schema of every kind declaring a set holding the parameter n, for an int; Matrix, declaring rows and cols; and
    Tree, whose parameter is for a reference to a tree."""
    S = txn.Schemas
    return txn.uid("par.declaring/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Object": S.OfObject, "Relation": S.OfRelation, "Union": S.OfUnion, "Intersection": S.OfIntersection,
                  "Container": S.OfContainer, "Pattern": S.OfPattern, "Matrix": S.OfObject, "Tree": S.OfObject})
        .spec([
            lambda _: _.Object.name("par.Object").parameters([txn.Parametrics.OfParameters(lambda ps: ps.parameters({
                "n": lambda d: d.parameter(lambda p: p.for_value([S.OfInt]))}))]),
            lambda _: _.Relation.name("par.Relation").links([lambda t: t.name("a"), lambda t: t.name("b")]).parameters([
                txn.Parametrics.OfParameters(lambda ps: ps.parameters({
                    "n": lambda d: d.parameter(lambda p: p.for_value([S.OfInt]))}))]),
            lambda _: _.Union.name("par.Union").parameters([txn.Parametrics.OfParameters(lambda ps: ps.parameters({
                "n": lambda d: d.parameter(lambda p: p.for_value([S.OfInt]))}))]).branches([
                lambda t: t.name("a").schema([S.OfStr]), lambda t: t.name("b").schema([S.OfStr])]),
            lambda _: _.Intersection.name("par.Intersection").parameters([txn.Parametrics.OfParameters(lambda ps: ps.parameters({
                "n": lambda d: d.parameter(lambda p: p.for_value([S.OfInt]))}))]).parts([
                lambda t: t.name("a").schema([S.OfStr]), lambda t: t.name("b").schema([S.OfStr])]),
            lambda _: _.Container.name("par.Container").parameters([txn.Parametrics.OfParameters(lambda ps: ps.parameters({
                "n": lambda d: d.parameter(lambda p: p.for_value([S.OfInt]))}))]).index([S.OfInt]).element([S.OfStr]),
            lambda _: _.Pattern.name("par.Pattern").parameters([txn.Parametrics.OfParameters(lambda ps: ps.parameters({
                "n": lambda d: d.parameter(lambda p: p.for_value([S.OfInt]))}))]),
            lambda _: _.Matrix.name("Matrix").parameters([txn.Parametrics.OfParameters(lambda ps: ps.parameters({
                "rows": lambda d: d.parameter(lambda p: p.for_value([S.OfInt])),
                "cols": lambda d: d.parameter(lambda p: p.description("How many columns").for_value([S.OfInt])),
            }))]),
            lambda _: _.Tree.name("Tree").ref(True).parameters([txn.Parametrics.OfParameters(lambda ps: ps.parameters({
                "root": lambda d: d.parameter(lambda p: p.for_object([_.Tree]))}))]),
        ])]))])


declared = B.Store()
declared.transaction(Parametrics.install)
declared.transaction(declaring)
for name, kind in [("par.Object", B.OfObject), ("par.Relation", B.OfRelation), ("par.Union", B.OfUnion),
                   ("par.Intersection", B.OfIntersection), ("par.Container", B.OfContainer), ("par.Pattern", B.OfPattern)]:
    defn = declared.registered(name)
    (held,) = kind.Builder(defn, False, declared).parameters  # one set, its parameters keyed by their names
    assert [e.name.get() for e in held.params.parameters] == ["n"] and Parametrics.validate(declared, defn) == []
(held,) = B.OfObject.Builder(declared.registered("Matrix"), False, declared).parameters
rows, cols = held.params.parameters  # in order
assert (rows.name.get(), cols.name.get()) == ("rows", "cols") and isinstance(cols.parameter, S.OfParameter.Data)
assert (rows.parameter.description, cols.parameter.description) == (None, "How many columns")
assert S.OfParameter.Data("w", "Its width") == S.OfParameter.Data("w", "Its width") != S.OfParameter.Data("w")  # by what it holds
assert [e.schema.name.get() for e in B.OfParameter.Builder(cols.parameter, False, declared).for_value] == ["Schemas.OfInt"]
(held,) = B.OfObject.Builder(declared.registered("Tree"), False, declared).parameters
(root,) = held.params.parameters
ends = B.OfParameter.Builder(root.parameter, False, declared)
assert [e.schema.name.get() for e in ends.for_object] == ["Tree"] and list(ends.for_value) == []  # for an object
log = Plain.ToPlain(declared).Snapshot()
declared.transaction(declaring)  # run again: found, nothing written
assert Plain.ToPlain(declared).Snapshot() == log
with declared.transaction():  # imperatively, the set built where the definition links it
    Vector = declared.Schemas.OfObject(lambda o: o.name("Vector").parameters([declared.Parametrics.OfParameters(
        lambda ps: ps.parameters({"n": lambda d: d.parameter(lambda p: p.for_value([declared.Schemas.OfInt]))}))]))
(held,) = B.OfObject.Builder(Vector, False, declared).parameters
assert [e.name.get() for e in held.params.parameters] == ["n"] and Parametrics.validate(declared, Vector) == []
assert Parametrics.validate(declared, declared.Schemas.OfInt.as_schema()) == []  # none declared: none to check

<a id="PAR-03"></a>
## PAR-03 · Validation checks a set's entries: reserved names, descriptions, a parameter for one of a value and an object, a pattern's names, and one set to one definition

In [ ]:
def flawed(txn):
    """A pattern whose set has a reserved name, a parameter for neither a value nor an object, one for both, and names
    its symbol and its variable have; and Shared and Twice, which share one set."""
    S = txn.Schemas
    return txn.uid("par.flawed/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Pattern": S.OfPattern, "Node": S.OfObject})
        .spec([
            lambda _: _.Node.name("par.Node").ref(True),
            lambda _: _.Pattern.name("par.Flawed").symbols({"x": _.Node}).variables({"w": S.OfInt}).parameters([
                txn.Parametrics.OfParameters(lambda ps: ps.parameters({
                    "$n": lambda d: d.parameter(lambda p: p.for_value([S.OfInt])),
                    "x": lambda d: d.parameter(lambda p: p.for_object([_.Node])),
                    "w": lambda d: d.parameter(lambda p: p),  # for neither
                    "b": lambda d: d.parameter(lambda p: p.for_value([S.OfInt]).for_object([_.Node])),  # for both
                }))]),
        ])]))])


checked = B.Store()
checked.transaction(Parametrics.install)
checked.transaction(flawed)
Flawed = checked.registered("par.Flawed")
assert Parametrics.validate(checked, Flawed) == [
    "name '$n' is reserved: names starting with '$' belong to the wire format",
    "parameter 'w' is for neither a value nor an object: a parameter is for one",
    "parameter 'b' is for both a value and an object: a parameter is for one",
    "names used as both symbol and parameter: ['x']",
    "names used as both variable and parameter: ['w']"]
assert Flawed.validate() == []  # a kind's own validate() does not read the extension's entries
(held,) = B.OfPattern.Builder(Flawed, False, checked).parameters
dollar = next(iter(held.params.parameters)).parameter
dollar.description = True  # data the builders refuse to build
assert Parametrics.validate(checked, Flawed)[1] == "parameter '$n': a description is text, got bool"
with raises(TypeError, match="property 'description' holds str, got bool"):
    with checked.transaction():
        checked.Schemas.OfParameter(lambda p: p.description(True))
with checked.transaction():  # one set linked by two definitions, and a definition linking two sets
    shared = checked.Parametrics.OfParameters(lambda ps: ps.parameters({
        "n": lambda d: d.parameter(lambda p: p.for_value([checked.Schemas.OfInt]))}))
    other = checked.Parametrics.OfParameters(lambda ps: ps)
    One = checked.Schemas.OfObject(lambda o: o.name("par.One").parameters([shared]))
    Two = checked.Schemas.OfObject(lambda o: o.name("par.Two").parameters([shared, other]))
assert Parametrics.validate(checked, One) == ["its set of parameters belongs to 2 definitions: a set belongs to one"]
assert Parametrics.validate(checked, Two) == ["it has 2 sets of parameters: a definition has one",
                                              "its set of parameters belongs to 2 definitions: a set belongs to one"]

<a id="PAR-04"></a>
## PAR-04 · Snapshots write a definition's set as the extension's entry and read it back; a store without Parametrics ignores what needs it

In [ ]:
def grid(txn):
    """Grid, declaring rows and cols, each for an int, its cells a container of floats."""
    S = txn.Schemas
    return txn.uid("par.grid/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Grid": S.OfObject})
        .spec([
            lambda _: _.Grid.name("Grid").parameters([txn.Parametrics.OfParameters(lambda ps: ps.parameters({
                "rows": lambda d: d.parameter(lambda p: p.for_value([S.OfInt])),
                "cols": lambda d: d.parameter(lambda p: p.description("Its width").for_value([S.OfInt])),
            }))]).properties({"cells": S.OfContainer(lambda c: c.index([S.OfInt]).element([S.OfFloat]))}),
        ])]))])


written = B.Store()
written.transaction(Parametrics.install)
written.transaction(grid)
log = Plain.ToPlain(written).Snapshot()
(entry,) = entries_in(spec_of(log, "Grid", 1), "parameters")  # the extension's entry, with the data: written once
assert entry == {"params": {"reference": {"symbol": "0"}}} and entries_in(spec_of(log, "Grid"), "parameters") == []
parameters = entries_in(spec_of(log, "0", 1), "parameters")  # the set's, each defining the parameter its entry links
assert [e["name"] for e in parameters] == ["cols", "rows"]  # data's keyed entries, written in the order of their keys
assert [given(e["parameter"]["def"]["mutations"]) for e in parameters] == [{"description": "Its width"}, {}]
back = B.Store()
back.transaction(Parametrics.install)  # a store with Parametrics installed
Plain.FromPlain(back).Snapshot(log)
assert Plain.ToPlain(back).Snapshot() == log  # read back, written again: the same
(held,) = B.OfObject.Builder(back.registered("Grid"), False, back).parameters
assert [e.name.get() for e in held.params.parameters] == ["cols", "rows"]
assert Parametrics.validate(back, back.registered("Grid")) == [] and back.registered("Grid") is not written.registered("Grid")
back.transaction(grid)  # its install, run again: found, nothing written
assert Plain.ToPlain(back).Snapshot() == log
bare, ignored = B.Store(), []  # a store without it reads it as a snapshot made with extensions it lacks
Plain.FromPlain(bare).Snapshot(log, ignored)
assert [i["reason"] for i in ignored] == ["Schemas.OfParameter has no adjacency 'for_value'"] * 2 + [
    "Schemas.OfObject has no adjacency 'parameters'"]
assert list(bare.registered("Grid").properties) == ["cells"] and "parameters" not in bare.Schemas.OfObject.as_schema().adjacencies

<a id="PAR-05"></a>
## PAR-05 · `extend` gives a reference object schema the adjacencies through which its objects bind; it refuses a store without Parametrics, a value object schema and a member's name taken; run again, it writes nothing

In [ ]:
def hardware(txn):
    """hw.Bus, a reference object declaring width, for an int, and host, for a hw.Chip; hw.Chip, which a binding may link
    (anonymously, as the object bound); hw.Pin, a value object; hw.Taken and hw.Linked, which declare a member of each
    name extend gives."""
    S = txn.Schemas
    return txn.uid("par.hardware/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Bus": S.OfObject, "Chip": S.OfObject, "Pin": S.OfObject, "Taken": S.OfObject, "Linked": S.OfObject,
                  "Wires": S.OfRelation})
        .spec([
            lambda _: _.Chip.name("hw.Chip").ref(True).adjacencies([
                lambda a: a.relation([txn.Parametrics.BindsObject]).me("obj_ref")]),
            lambda _: _.Bus.name("hw.Bus").ref(True).properties({"label": S.OfStr}).parameters([
                txn.Parametrics.OfParameters(lambda ps: ps.parameters({
                    "width": lambda d: d.parameter(lambda p: p.for_value([S.OfInt])),
                    "host": lambda d: d.parameter(lambda p: p.for_object([_.Chip])),
                }))]),
            lambda _: _.Pin.name("hw.Pin").properties({"number": S.OfInt}),
            lambda _: _.Wires.name("hw.Wires").links([lambda t: t.name("from"), lambda t: t.name("to")]),
            lambda _: _.Taken.name("hw.Taken").ref(True).properties({"bound_values": S.OfStr}),
            lambda _: _.Linked.name("hw.Linked").ref(True).adjacencies({
                "bound_objects": lambda a: a.relation([_.Wires]).me("from")}),
        ])]))])


bare = B.Store()
with raises(LookupError, match="Parametrics.extend: the store has no Parametrics installed; install it first, "
                                "store.transaction(Parametrics.install)"):
    bare.transaction(lambda t: Parametrics.extend(t, [t.Schemas.OfObject]))
hw = B.Store()
hw.transaction(Parametrics.install)
hw.transaction(hardware)
Bus, BindsValue, BindsObject = (hw.registered(name) for name in (
    "hw.Bus", "Parametrics.BindsValue", "Parametrics.BindsObject"))
assert "bound_values" not in Bus.adjacencies  # not extended: its objects bind nothing
hw.transaction(lambda t: Parametrics.extend(t, [t.hw.Bus]))
assert [(name, a.relation, a.me) for name, a in Bus.adjacencies.items()] == [
    ("bound_values", BindsValue, "object"), ("bound_objects", BindsObject, "object")]
log = Plain.ToPlain(hw).Snapshot()
hw.transaction(lambda t: Parametrics.extend(t, [t.hw.Bus, hw.registered("hw.Bus")]))  # run again: nothing written
assert Plain.ToPlain(hw).Snapshot() == log
with raises(TypeError, match="Parametrics.extend takes reference object schemas: 'hw.Pin' is a value object schema, "
                             "whose objects are never linked, so that none binds a parameter"):
    hw.transaction(lambda t: Parametrics.extend(t, [t.hw.Pin]))
with raises(TypeError, match="Parametrics.extend takes reference object schemas: 'Schemas.OfInt' is not one"):
    hw.transaction(lambda t: Parametrics.extend(t, [t.Schemas.OfInt]))
with raises(TypeError, match="Parametrics.extend takes reference object schemas: 'hw.Wires' is not one"):
    hw.transaction(lambda t: Parametrics.extend(t, [t.hw.Wires]))
with raises(TypeError, match="Parametrics.extend takes reference object schemas: 'nothing' is not one"):
    hw.transaction(lambda t: Parametrics.extend(t, ["nothing"]))
with raises(TypeError, match="Parametrics.extend: 'hw.Taken' already declares a member 'bound_values'"):
    hw.transaction(lambda t: Parametrics.extend(t, [t.hw.Taken]))
with raises(TypeError, match="Parametrics.extend: 'hw.Linked' already declares a member 'bound_objects'"):
    hw.transaction(lambda t: Parametrics.extend(t, [t.hw.Linked]))
assert Plain.ToPlain(hw).Snapshot() == log  # each refused whole: nothing written


<a id="PAR-06"></a>
## PAR-06 · An object binds a value and an object to the parameters its schema declares; a parameter it leaves unbound stays unbound

In [ ]:
with hw.transaction():
    (held,) = B.OfObject.Builder(Bus, False, hw).parameters
    width, host = (e.parameter for e in held.params.parameters)
    chip = hw.hw.Chip(lambda c: c)
    bus = hw.hw.Bus(lambda b: b.label("data")
                    .bound_values([lambda e: e.param(width).value(8)])
                    .bound_objects(lambda a: a.insert(lambda e: e.param(host).obj_ref(chip))))
    (value,), (obj,) = bus.bound_values, bus.bound_objects
    assert value.param is width and value.value.get() == 8  # its value an entry property
    assert obj.param is host and obj.obj_ref == chip  # its object a third link
    narrow = hw.hw.Bus(lambda b: b.label("narrow").bound_values([lambda e: e.param(width).value(1)]))
    assert list(narrow.bound_objects) == []  # host left unbound: it stays unbound
    loose = hw.hw.Bus(lambda b: b.label("loose"))
    assert list(loose.bound_values) == list(loose.bound_objects) == []  # binding is not required
    assert Parametrics.validate(hw, bus) == Parametrics.validate(hw, narrow) == Parametrics.validate(hw, loose) == []
    assert Validators.Validate(hw)(Bus, bus) == []  # each link filled by a schema whose adjacency says so
    board = hw.hw.Bus(lambda b: b.label("board"))
    n = hw.References.OfParameter(lambda p: p.name("n"))  # a parameter reference in the place of the parameter
    hw.builder("hw.Bus", board).bound_values([lambda e: e.param(n).value(3)]).update()  # what it binds is unknown here
    assert Parametrics.validate(hw, board) == []


<a id="PAR-07"></a>
## PAR-07 · A binding of a parameter its object's schema does not declare refuses its transaction when it commits; validation reports one already held, and a snapshot reads it back as written

In [ ]:
def wiring(txn):
    """hw.Wire, declaring depth, and hw.Top, a singleton declaring n and m."""
    S = txn.Schemas
    return txn.uid("par.wiring/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Wire": S.OfObject, "Top": S.OfObject})
        .spec([
            lambda _: _.Wire.name("hw.Wire").ref(True).parameters([txn.Parametrics.OfParameters(lambda ps: ps.parameters({
                "depth": lambda d: d.parameter(lambda p: p.for_value([S.OfInt]))}))]),
            lambda _: _.Top.name("hw.Top").ref(True).singleton("hw.Top").parameters([
                txn.Parametrics.OfParameters(lambda ps: ps.parameters({
                    "n": lambda d: d.parameter(lambda p: p.for_value([S.OfInt])),
                    "m": lambda d: d.parameter(lambda p: p.for_value([S.OfInt]))}))]),
        ])]))])


hw.transaction(wiring)
hw.transaction(lambda t: Parametrics.extend(t, [t.hw.Wire, t.hw.Top]))
log = Plain.ToPlain(hw).Snapshot()
with hw.transaction():
    (held,) = B.OfObject.Builder(hw.registered("hw.Wire"), False, hw).parameters
    (depth,) = (e.parameter for e in held.params.parameters)
    free = hw.Schemas.OfParameter(lambda p: p.for_value([hw.Schemas.OfInt]))  # in no set
with raises(ValueError, match="an entry of 'Parametrics.BindsValue' binds parameter 'depth', which 'hw.Bus' does not "
                              "declare: an object binds only the parameters its schema declares"):
    with hw.transaction():
        hw.hw.Bus(lambda b: b.bound_values([lambda e: e.param(depth).value(8)]))  # a hw.Wire's: written, and
                                                                                  # refused at commit, as a unique clause is
with raises(ValueError, match="an entry of 'Parametrics.BindsObject' binds a parameter no set of parameters defines, "
                              "which 'hw.Bus' does not declare"):
    with hw.transaction():
        hw.hw.Bus(lambda b: b.bound_objects(lambda a: a.insert(lambda e: e.param(free).obj_ref(hw.hw.Chip(lambda c: c)))))
with raises(ValueError, match="link 'param' is not set"):  # a binding names its parameter
    with hw.transaction():
        hw.hw.Bus(lambda b: b.bound_values(lambda a: a.insert(lambda e: e.value(8))))
assert Plain.ToPlain(hw).Snapshot() == log  # each refused whole
with hw.transaction():
    (held,) = B.OfObject.Builder(hw.registered("hw.Top"), False, hw).parameters
    n, m = (e.parameter for e in held.params.parameters)
    hw.builder("hw.Top", hw.singleton("hw.Top")).bound_values([lambda e: e.param(n).value(3)]).update()
with hw.transaction():  # an object that binds already binds another: the binding inserted is checked
    hw.builder("hw.Top", hw.singleton("hw.Top")).bound_values(lambda a: a.insert(lambda e: e.param(m).value(4))).update()
    assert [e.value.get() for e in hw.singleton("hw.Top").bound_values] == [3, 4]
with hw.transaction():  # the set's declarations removed: the bindings held stay
    (held,) = B.OfObject.Builder(hw.registered("hw.Top"), False, hw).parameters
    hw.builder("Parametrics.OfParameters", held.params._target()).parameters(lambda a: a.clear()).update()
with hw.transaction():  # n and m no longer declared: the bindings held are reported, not refused
    assert Parametrics.validate(hw, hw.singleton("hw.Top")) == [
        "an entry of 'Parametrics.BindsValue' binds a parameter no set of parameters defines, which 'hw.Top' does not "
        "declare: an object binds only the parameters its schema declares"] * 2
written = Plain.ToPlain(hw).Snapshot()  # the bindings, written as any entries are
back = B.Store()
back.transaction(Parametrics.install)
Plain.FromPlain(back).Snapshot(written)  # read back as written: not refused
assert Plain.ToPlain(back).Snapshot() == written
with back.transaction():
    assert [e.value.get() for e in back.singleton("hw.Top").bound_values] == [3, 4]
    assert len(Parametrics.validate(back, back.singleton("hw.Top"))) == 2


<a id="PAR-08"></a>
## PAR-08 · A schema binds none of its parameters: no kind of schema has an adjacency to a binding, a binding links an `object`, and extending a schema of schemas is refused

In [ ]:
for kind in KINDS:  # each kind holds its set of parameters, and no binding
    schema = getattr(hw.Schemas, kind).as_schema()
    assert all(a.relation not in (BindsValue, BindsObject)
               for a in (*schema.adjacencies.values(), *schema.anonymous))
assert [link for link in BindsValue.entry.links] == ["object", "param"] and "defn" not in BindsObject.entry.links
with raises(TypeError, match="Parametrics.extend: 'Schemas.OfObject' is a schema of schemas, whose objects are schemas: "
                             "a schema declares its parameters and binds none of them"):
    hw.transaction(lambda t: Parametrics.extend(t, [t.Schemas.OfObject]))
assert "bound_values" not in hw.Schemas.OfObject.as_schema().adjacencies
